In [1]:
print("Here begins the Bayesian networks notebook.")

Here begins the Bayesian networks notebook.


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
df_bn = pd.read_csv("../../data/processed/life_satisfaction_canadian_survey_bn_clean.csv")

df_bn.head()

,Health_region_ grouped,Gender,Marital_status,Household,Age,Worked_job_business,Edu_level,Gen_health_state,Life_satisfaction,Mental_health_state,Stress_level,Work_stress,Sense_belonging,Weight_state,BMI_12_17,BMI_18_above,Sleep_apnea,High_BP,High_cholestrol,Diabetic,Fatigue_syndrome,Mood_disorder,Anxiety_disorder,Respiratory_chronic_con,Musculoskeletal_con,Cardiovascular_con,Health_utility_indx,Pain_status,Act_improve_health,Fruit_veg_con,Smoked,Tobaco_use,weekly_alcohol,Cannabies_use,Drug_use,Total_active_time,Total_physical_act_time,Other_physical_act_time,Physical_vigorous_act_time,Work_hours,working_status,Aboriginal_identity,Birth_country,Immigrant,Insurance_cover,Food_security,Income_source,Total_income
0,47906,2,1.0,2.0,3,1.0,3.0,3.0,9.0,3.0,2.0,2.0,2.0,3.0,NaN,2.0,2.0,2.0,2,2.0,2.0,2.0,2.0,2.0,2.0,2.0,1.0,2.0,NaN,bin_0,NaN,NaN,NaN,2.0,2.0,0.0,0.0,60.0,10.0,38.0,1.0,2.0,1.0,2.0,1.0,0.0,1.0,5.0
1,47906,1,1.0,2.0,5,NaN,2.0,3.0,4.0,3.0,3.0,NaN,3.0,1.0,NaN,2.0,1.0,1.0,2,2.0,2.0,1.0,2.0,2.0,2.0,2.0,NaN,1.0,NaN,bin_0,NaN,NaN,NaN,2.0,2.0,0.0,0.0,0.0,0.0,NaN,NaN,2.0,1.0,2.0,1.0,0.0,2.0,4.0
2,59914,2,2.0,1.0,5,NaN,1.0,2.0,7.0,3.0,3.0,NaN,2.0,1.0,NaN,2.0,2.0,1.0,2,1.0,2.0,2.0,2.0,1.0,1.0,2.0,1.0,1.0,NaN,bin_1,NaN,2.0,NaN,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,1.0,2.0,6.0,NaN,2.0,2.0
3,13904,1,2.0,1.0,5,NaN,1.0,3.0,8.0,3.0,3.0,NaN,2.0,1.0,NaN,2.0,2.0,1.0,2,2.0,2.0,2.0,2.0,2.0,1.0,2.0,2.0,1.0,NaN,bin_1,NaN,NaN,NaN,2.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,1.0,2.0,6.0,0.0,2.0,3.0
4,46903,1,2.0,1.0,4,2.0,3.0,5.0,0.0,5.0,4.0,NaN,3.0,3.0,NaN,2.0,2.0,1.0,2,NaN,2.0,2.0,2.0,NaN,1.0,NaN,1.0,2.0,NaN,bin_1,NaN,NaN,NaN,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,1.0,2.0,2.0,0.0,2.0,1.0


In [2]:
# Inspect shape, data types, and missing values
print(df_bn.shape)
print(df_bn.dtypes)

print("\nMissing values per column:")
print(df_bn.isna().sum().sort_values(ascending=False))

(108252, 48)
Health_region_ grouped          int64
Gender                          int64
Marital_status                float64
Household                     float64
Age                             int64
Worked_job_business           float64
Edu_level                     float64
Gen_health_state              float64
Life_satisfaction             float64
Mental_health_state           float64
Stress_level                  float64
Work_stress                   float64
Sense_belonging               float64
Weight_state                  float64
BMI_12_17                     float64
BMI_18_above                  float64
Sleep_apnea                   float64
High_BP                       float64
High_cholestrol                 int64
Diabetic                      float64
Fatigue_syndrome              float64
Mood_disorder                 float64
Anxiety_disorder              float64
Respiratory_chronic_con       float64
Musculoskeletal_con           float64
Cardiovascular_con            float64

In [3]:
# Convert all columns to string (discrete states)
# Missing values are preserved as NaN
for col in df_bn.columns:
    df_bn[col] = df_bn[col].astype("string")

print(df_bn.dtypes)

Health_region_ grouped        string
Gender                        string
Marital_status                string
Household                     string
Age                           string
Worked_job_business           string
Edu_level                     string
Gen_health_state              string
Life_satisfaction             string
Mental_health_state           string
Stress_level                  string
Work_stress                   string
Sense_belonging               string
Weight_state                  string
BMI_12_17                     string
BMI_18_above                  string
Sleep_apnea                   string
High_BP                       string
High_cholestrol               string
Diabetic                      string
Fatigue_syndrome              string
Mood_disorder                 string
Anxiety_disorder              string
Respiratory_chronic_con       string
Musculoskeletal_con           string
Cardiovascular_con            string
Health_utility_indx           string
P

In [4]:
# Check the number of unique states per variable
cardinality = pd.DataFrame({
    "Variable": df_bn.columns,
    "Num_States": [df_bn[col].nunique(dropna=True) for col in df_bn.columns]
}).sort_values("Num_States", ascending=False)

cardinality

,Variable,Num_States
35,Total_active_time,211
36,Total_physical_act_time,208
37,Other_physical_act_time,202
38,Physical_vigorous_act_time,156
0,Health_region_ grouped,91
32,weekly_alcohol,80
30,Smoked,51
39,Work_hours,50
8,Life_satisfaction,11
18,High_cholestrol,5


In [11]:
# Create the BN modeling dataset
df_bn_model = df_bn.copy()

# Drop high-cardinality region identifier
df_bn_model = df_bn_model.drop(columns=["Health_region_ grouped"], errors="ignore")

# Discretize time variables into bins
time_cols = [
    "Total_active_time",
    "Total_physical_act_time",
    "Other_physical_act_time",
    "Physical_vigorous_act_time"
]

bin_name_map = {
    2: ["low", "high"],
    3: ["low", "medium", "high"],
    4: ["low", "medium_low", "medium_high", "high"],
    5: ["very_low", "low", "medium", "high", "very_high"]
}

for col in time_cols:
    if col in df_bn_model.columns:
        s = pd.to_numeric(df_bn_model[col], errors="coerce")

        if s.nunique(dropna=True) > 20:
            # First create numeric bins, allowing qcut to drop duplicate edges
            binned = pd.qcut(
                s,
                q=5,
                labels=False,
                duplicates="drop"
            )

            # Convert to nullable integer so NaNs are preserved
            binned = binned.astype("Int64")

            # Determine how many bins were actually created
            n_bins = binned.dropna().nunique()

            # Get appropriate labels
            labels = bin_name_map.get(
                n_bins,
                [f"bin_{i}" for i in range(n_bins)]
            )

            # Map numeric bins to string labels
            df_bn_model[f"{col}_binned"] = binned.map(
                lambda x: labels[int(x)] if pd.notna(x) else pd.NA
            ).astype("string")

            # Drop original high-cardinality column
            df_bn_model = df_bn_model.drop(columns=[col])

print(df_bn_model.shape)
print(df_bn_model.columns.tolist())

(108252, 47)
['Gender', 'Marital_status', 'Household', 'Age', 'Worked_job_business', 'Edu_level', 'Gen_health_state', 'Life_satisfaction', 'Mental_health_state', 'Stress_level', 'Work_stress', 'Sense_belonging', 'Weight_state', 'BMI_12_17', 'BMI_18_above', 'Sleep_apnea', 'High_BP', 'High_cholestrol', 'Diabetic', 'Fatigue_syndrome', 'Mood_disorder', 'Anxiety_disorder', 'Respiratory_chronic_con', 'Musculoskeletal_con', 'Cardiovascular_con', 'Health_utility_indx', 'Pain_status', 'Act_improve_health', 'Fruit_veg_con', 'Smoked', 'Tobaco_use', 'weekly_alcohol', 'Cannabies_use', 'Drug_use', 'Work_hours', 'working_status', 'Aboriginal_identity', 'Birth_country', 'Immigrant', 'Insurance_cover', 'Food_security', 'Income_source', 'Total_income', 'Total_active_time_binned', 'Total_physical_act_time_binned', 'Other_physical_act_time_binned', 'Physical_vigorous_act_time_binned']


In [12]:
for col in df_bn_model.columns:
    if col.endswith("_binned"):
        print(f"\n{col}:")
        print(df_bn_model[col].value_counts(dropna=False))


Total_active_time_binned:
Total_active_time_binned
<NA>    87474
low     16873
high     3905
Name: count, dtype: Int64

Total_physical_act_time_binned:
Total_physical_act_time_binned
<NA>    87491
low     16678
high     4083
Name: count, dtype: Int64

Other_physical_act_time_binned:
Other_physical_act_time_binned
<NA>    87520
low     16679
high     4053
Name: count, dtype: Int64

Physical_vigorous_act_time_binned:
Physical_vigorous_act_time_binned
<NA>     87728
bin_0    20524
Name: count, dtype: Int64


In [13]:
df_bn_model = df_bn_model.drop(columns=[
    "Total_active_time_binned",
    "Total_physical_act_time_binned",
    "Other_physical_act_time_binned",
    "Physical_vigorous_act_time_binned"
], errors="ignore")

In [14]:
# Check number of states per variable
cardinality = pd.DataFrame({
    "Variable": df_bn_model.columns,
    "Num_States": [df_bn_model[col].nunique(dropna=True) for col in df_bn_model.columns]
}).sort_values("Num_States", ascending=False)

cardinality

,Variable,Num_States
31,weekly_alcohol,80
29,Smoked,51
34,Work_hours,50
7,Life_satisfaction,11
9,Stress_level,5
6,Gen_health_state,5
42,Total_income,5
17,High_cholestrol,5
10,Work_stress,5
3,Age,5


In [16]:
# Drop weekly_alcohol completely - many invalid values and high cardinality

# Drop weekly_alcohol
df_bn_model = df_bn_model.drop(columns=["weekly_alcohol"], errors="ignore")

# Discretize high-cardinality numeric variables
high_card_cols = ["Smoked", "Work_hours"]

for col in high_card_cols:
    if col in df_bn_model.columns:
        s = pd.to_numeric(df_bn_model[col], errors="coerce")

        if s.nunique(dropna=True) > 20:
            binned = pd.qcut(
                s,
                q=5,
                labels=False,
                duplicates="drop"
            )

            binned = binned.astype("Int64")

            n_bins = binned.dropna().nunique()

            label_pool = ["very_low", "low", "medium", "high", "very_high"]
            labels = label_pool[:n_bins]

            df_bn_model[f"{col}_binned"] = binned.map(
                lambda x: labels[int(x)] if pd.notna(x) else pd.NA
            ).astype("string")

            # Drop original high-cardinality column
            df_bn_model = df_bn_model.drop(columns=[col])

print("BN modeling dataset shape:", df_bn_model.shape)
print("\nColumns:")
print(df_bn_model.columns.tolist())

print("\nDiscretized variables:")
for col in df_bn_model.columns:
    if col.endswith("_binned"):
        print(f"\n{col}:")
        print(df_bn_model[col].value_counts(dropna=False))

BN modeling dataset shape: (108252, 42)

Columns:
['Gender', 'Marital_status', 'Household', 'Age', 'Worked_job_business', 'Edu_level', 'Gen_health_state', 'Life_satisfaction', 'Mental_health_state', 'Stress_level', 'Work_stress', 'Sense_belonging', 'Weight_state', 'BMI_12_17', 'BMI_18_above', 'Sleep_apnea', 'High_BP', 'High_cholestrol', 'Diabetic', 'Fatigue_syndrome', 'Mood_disorder', 'Anxiety_disorder', 'Respiratory_chronic_con', 'Musculoskeletal_con', 'Cardiovascular_con', 'Health_utility_indx', 'Pain_status', 'Act_improve_health', 'Fruit_veg_con', 'Tobaco_use', 'Cannabies_use', 'Drug_use', 'working_status', 'Aboriginal_identity', 'Birth_country', 'Immigrant', 'Insurance_cover', 'Food_security', 'Income_source', 'Total_income', 'Smoked_binned', 'Work_hours_binned']

Discretized variables:

Smoked_binned:
Smoked_binned
<NA>         96158
low           3007
very_low      2873
medium        2457
very_high     1994
high          1763
Name: count, dtype: Int64

Work_hours_binned:
Work_hou

In [17]:
# Check number of states per variable
cardinality = pd.DataFrame({
    "Variable": df_bn_model.columns,
    "Num_States": [df_bn_model[col].nunique(dropna=True) for col in df_bn_model.columns]
}).sort_values("Num_States", ascending=False)

cardinality

,Variable,Num_States
7,Life_satisfaction,11
3,Age,5
9,Stress_level,5
8,Mental_health_state,5
6,Gen_health_state,5
36,Insurance_cover,5
39,Total_income,5
40,Smoked_binned,5
17,High_cholestrol,5
10,Work_stress,5


In [18]:
# Optional final cleanup for BN stability
df_bn_model = df_bn_model.drop(columns=["Smoked_binned"], errors="ignore")

print(df_bn_model.shape)
print(df_bn_model.columns.tolist())

(108252, 41)
['Gender', 'Marital_status', 'Household', 'Age', 'Worked_job_business', 'Edu_level', 'Gen_health_state', 'Life_satisfaction', 'Mental_health_state', 'Stress_level', 'Work_stress', 'Sense_belonging', 'Weight_state', 'BMI_12_17', 'BMI_18_above', 'Sleep_apnea', 'High_BP', 'High_cholestrol', 'Diabetic', 'Fatigue_syndrome', 'Mood_disorder', 'Anxiety_disorder', 'Respiratory_chronic_con', 'Musculoskeletal_con', 'Cardiovascular_con', 'Health_utility_indx', 'Pain_status', 'Act_improve_health', 'Fruit_veg_con', 'Tobaco_use', 'Cannabies_use', 'Drug_use', 'working_status', 'Aboriginal_identity', 'Birth_country', 'Immigrant', 'Insurance_cover', 'Food_security', 'Income_source', 'Total_income', 'Work_hours_binned']
